In [0]:
bronze_df = spark.table("workspace.default.bronze_calls")

In [0]:
from pyspark.sql.functions import *

silver_df = (
    bronze_df
        .withColumn("agent", col("conversation.agent"))
        .withColumn("customer_text", col("conversation.customer_text"))
        .drop("conversation")
)

In [0]:
display(silver_df)

call_id,customer_id,call_timestamp,duration_seconds,sentiment,agent,customer_text
06ae2680-e21f-4e5e-8b0e-abb3c4564655,8,2026-07-11T20:12:22.727Z,478,Negative,Agent_12,Problem not resolved
0d9d63f5-4d9e-4bc7-83e6-733ec78248bf,87,2026-07-11T20:20:07.573Z,650,Negative,Agent_16,Bad experience
0df319b5-6c7c-438c-a97e-1dc8d73cd9cb,80,2026-07-11T20:22:20.535Z,700,Neutral,Agent_19,Need more information
0fa51155-219c-49c8-8c02-2695fceb8604,21,2026-07-11T20:18:25.540Z,864,Neutral,Agent_10,Waiting for update
10c8e57d-8a5c-4cbe-9b61-09b47b617488,45,2026-07-11T20:20:53.456Z,648,Positive,Agent_3,Very happy with service
1459364a-bcaa-4490-ae81-4d33f5b65719,43,2026-07-11T20:22:15.447Z,593,Neutral,Agent_5,Call transferred
14ff805f-a6a0-4cfc-9784-7a08921deb2f,75,2026-07-11T20:16:23.080Z,244,Negative,Agent_16,Bad experience
164f9246-289c-421f-aacf-9ff8edd57076,93,2026-07-11T20:13:34.225Z,1197,Negative,Agent_4,Bad experience
17830b84-c92f-42c8-8c31-a05dda96d1b6,66,2026-07-11T20:12:53.332Z,160,Neutral,Agent_7,Call transferred
1841aba7-319c-4a7c-a9ac-962ecf322da0,10,2026-07-11T20:19:06.373Z,1057,Neutral,Agent_16,Need more information


In [0]:
silver_df = silver_df.dropDuplicates(["call_id"])

In [0]:
from pyspark.sql.functions import to_timestamp

silver_df = silver_df.withColumn(
    "event_time",
    to_timestamp("call_timestamp")
)

In [0]:
display(silver_df)

call_id,customer_id,call_timestamp,duration_seconds,sentiment,agent,customer_text,event_time
06ae2680-e21f-4e5e-8b0e-abb3c4564655,8,2026-07-11T20:12:22.727Z,478,Negative,Agent_12,Problem not resolved,2026-07-11T20:12:22.727Z
0d9d63f5-4d9e-4bc7-83e6-733ec78248bf,87,2026-07-11T20:20:07.573Z,650,Negative,Agent_16,Bad experience,2026-07-11T20:20:07.573Z
0df319b5-6c7c-438c-a97e-1dc8d73cd9cb,80,2026-07-11T20:22:20.535Z,700,Neutral,Agent_19,Need more information,2026-07-11T20:22:20.535Z
0fa51155-219c-49c8-8c02-2695fceb8604,21,2026-07-11T20:18:25.540Z,864,Neutral,Agent_10,Waiting for update,2026-07-11T20:18:25.540Z
10c8e57d-8a5c-4cbe-9b61-09b47b617488,45,2026-07-11T20:20:53.456Z,648,Positive,Agent_3,Very happy with service,2026-07-11T20:20:53.456Z
1459364a-bcaa-4490-ae81-4d33f5b65719,43,2026-07-11T20:22:15.447Z,593,Neutral,Agent_5,Call transferred,2026-07-11T20:22:15.447Z
14ff805f-a6a0-4cfc-9784-7a08921deb2f,75,2026-07-11T20:16:23.080Z,244,Negative,Agent_16,Bad experience,2026-07-11T20:16:23.080Z
164f9246-289c-421f-aacf-9ff8edd57076,93,2026-07-11T20:13:34.225Z,1197,Negative,Agent_4,Bad experience,2026-07-11T20:13:34.225Z
17830b84-c92f-42c8-8c31-a05dda96d1b6,66,2026-07-11T20:12:53.332Z,160,Neutral,Agent_7,Call transferred,2026-07-11T20:12:53.332Z
1841aba7-319c-4a7c-a9ac-962ecf322da0,10,2026-07-11T20:19:06.373Z,1057,Neutral,Agent_16,Need more information,2026-07-11T20:19:06.373Z


In [0]:
silver_table = "workspace.default.silver_calls"

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM workspace.default.silver_calls
    LIMIT 10
    """)
)

call_id,customer_id,call_timestamp,duration_seconds,sentiment,agent,customer_text,event_time
0df319b5-6c7c-438c-a97e-1dc8d73cd9cb,80,2026-07-11T20:22:20.535Z,700,Neutral,Agent_19,Need more information,2026-07-11T20:22:20.535Z
1841aba7-319c-4a7c-a9ac-962ecf322da0,10,2026-07-11T20:19:06.373Z,1057,Neutral,Agent_16,Need more information,2026-07-11T20:19:06.373Z
30726cac-1b61-490a-8680-1af9ffa45f6a,32,2026-07-11T20:17:54.953Z,602,Positive,Agent_7,Issue resolved successfully,2026-07-11T20:17:54.953Z
33a79673-9c65-4fa5-a956-d3b48f40dd07,53,2026-07-11T20:13:18.938Z,1095,Negative,Agent_17,Very disappointed,2026-07-11T20:13:18.938Z
55d9169c-04f4-4bae-9eef-1ccf13e0fc54,82,2026-07-11T20:13:08.701Z,494,Neutral,Agent_2,Need more information,2026-07-11T20:13:08.701Z
6a4da19d-c4a2-485a-856c-bab7bdfeb805,99,2026-07-11T20:19:57.393Z,717,Negative,Agent_7,Problem not resolved,2026-07-11T20:19:57.393Z
8aa1f8ba-f28d-4721-8c0f-a76bc900d94e,5,2026-07-11T20:11:11.074Z,719,Negative,Agent_13,Bad experience,2026-07-11T20:11:11.074Z
9cd668c2-a393-4286-8269-86856d8c603c,58,2026-07-11T20:13:13.801Z,715,Negative,Agent_10,Bad experience,2026-07-11T20:13:13.801Z
a16658ee-4cd9-435b-b041-f41019d81fc9,29,2026-07-11T20:13:29.130Z,276,Negative,Agent_16,Very disappointed,2026-07-11T20:13:29.130Z
ba368f9d-2e55-4b99-a4b6-c957c1d68b0a,67,2026-07-11T20:15:37.057Z,1132,Negative,Agent_18,Bad experience,2026-07-11T20:15:37.057Z
